# 第12章 部品を組み立てる：Transformer

話し合い（Attention）と、ひとりで考える時間（MLP）を、何層も積み重ねる

教科書のページ：https://sakunokyumei.github.io/llm-research-textbook/12-transformer.html

上のセルから順に、再生ボタン（▶）か `Shift + Enter` で実行します。説明は教科書のページにあります。


## 準備


In [ ]:
import re, math, urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

def load_soseki(file_id="773_ruby_5968"):
    url = ("https://raw.githubusercontent.com/aozorahack/aozorabunko_text/master/"
           f"cards/000148/files/{file_id}/{file_id}.txt")
    text = urllib.request.urlopen(url).read().decode("cp932")
    text = text.split("-------------------------------------------------------")[2]
    text = text.split("底本：")[0]
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"※?［＃.*?］", "", text)
    text = text.replace("｜", "").replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

text = load_soseki()
chars = sorted(set(text))
V = len(chars)
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text])
n = int(len(data) * 0.9)
train_data, val_data = data[:n], data[n:]

def get_batch(data, batch_size, T, g):
    ix = torch.randint(0, len(data) - T, (batch_size,), generator=g)
    x = torch.stack([data[i:i + T] for i in ix])
    y = torch.stack([data[i + 1:i + T + 1] for i in ix])
    return x, y

## 部品1：Head（おさらい）


In [ ]:
class Head(nn.Module):
    """Attentionヘッド1つ（第11章）"""

    def __init__(self, n_embd, head_size, block_size, dropout):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        scores = q @ k.transpose(-2, -1) / math.sqrt(k.shape[-1])
        scores = scores.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        self.wei = F.softmax(scores, dim=-1)
        return self.dropout(self.wei) @ v

## 部品2：マルチヘッドAttention


In [ ]:
class MultiHeadAttention(nn.Module):
    """ヘッドを何個も並べて、結果をつなげる"""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        head_size = n_embd // n_head
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size, dropout) for _ in range(n_head)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

mha = MultiHeadAttention(n_embd=64, n_head=4, block_size=8, dropout=0.0)
x = torch.randn(2, 8, 64)
print(mha(x).shape)

## 部品3：フィードフォワード（考える時間）


In [ ]:
class FeedForward(nn.Module):
    """各位置で、ひとりで「考える」MLP"""

    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

## 部品4：Block（残差接続とLayerNorm）


In [ ]:
class Block(nn.Module):
    """Transformerブロック：話し合い（Attention）→ 考える（FeedForward）"""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffwd = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

## 部品5：GPT（全体）


In [ ]:
class GPT(nn.Module):
    def __init__(self, vocab_size, block_size=128, n_embd=128, n_head=4, n_layer=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if isinstance(module, nn.Linear) and module.bias is not None:
            nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        x = self.blocks(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]          # 窓に入る分だけ
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature      # 最後の位置の点数
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

## 動かしてみる


In [ ]:
torch.manual_seed(0)
model = GPT(V, block_size=64, n_embd=64, n_head=4, n_layer=2, dropout=0.0)
total = sum(p.numel() for p in model.parameters())
print(f"パラメータの数：{total:,}")
groups = {"埋め込み（文字・位置）": 0, "Transformerブロック": 0, "最後のLayerNorm＋出力層": 0}
for name, p in model.named_parameters():
    if name.startswith(("tok_emb", "pos_emb")):
        groups["埋め込み（文字・位置）"] += p.numel()
    elif name.startswith("blocks"):
        groups["Transformerブロック"] += p.numel()
    else:
        groups["最後のLayerNorm＋出力層"] += p.numel()
for k, v in groups.items():
    print(f"  {k}：{v:,}（{v / total:.0%}）")   # :, は3桁ごとのカンマ、:.0% はパーセント表示

In [ ]:
g = torch.Generator().manual_seed(0)
x, y = get_batch(train_data, 8, 64, g)
_, loss = model(x, y)
print("最初の損失：", round(loss.item(), 3), " log V：", round(math.log(V), 3))

## 1バッチで丸暗記させるテスト


In [ ]:
torch.manual_seed(0)
model = GPT(V, block_size=64, n_embd=64, n_head=4, n_layer=2, dropout=0.0)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
x, y = get_batch(train_data, 8, 64, torch.Generator().manual_seed(0))   # この1バッチだけ
for step in range(301):
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 100 == 0:
        print(f"ステップ {step}：損失 {loss.item():.3f}")

## 『こころ』で少しだけ学習させる


In [ ]:
@torch.no_grad()
def val_loss(model, T):
    model.eval()
    starts = range(0, len(val_data) - T - 1, T)
    xs = torch.stack([val_data[i:i + T] for i in starts])
    ys = torch.stack([val_data[i + 1:i + T + 1] for i in starts])
    loss = model(xs, ys)[1].item()
    model.train()
    return loss

torch.manual_seed(0)
g = torch.Generator().manual_seed(0)
model = GPT(V, block_size=64, n_embd=64, n_head=4, n_layer=2, dropout=0.1)
opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
for step in range(1501):
    x, y = get_batch(train_data, 32, 64, g)
    _, loss = model(x, y)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 500 == 0:
        print(f"ステップ {step:4d}：訓練 {loss.item():.3f}  検証 {val_loss(model, 64):.3f}")

In [ ]:
model.eval()
torch.manual_seed(0)
start = torch.tensor([[stoi[c] for c in "私はその人を"]])
out = model.generate(start, max_new_tokens=80, temperature=0.8)
print("".join(itos[i] for i in out[0].tolist()))